#Módulo de Trabajo Infantil - MTI - GEIH - 2019

Generar datos cuantitativos sobre las actividades de los niños, niñas y adolescentes (incluidas las escolares, las económicas y las no económicas) que permita hacer el seguimiento a los principales indicadores de trabajo infantil.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import LongType, DoubleType, BooleanType, IntegerType, StringType
from functools import reduce

## Lectura y Descripción del Dataset

In [0]:
# Módulo de Trabajo Infantil - MTI - GEIH - 2019 https://microdatos.dane.gov.co/index.php/catalog/664/data-dictionary
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "UTF-8") \
    .option("sep", ";") \
    .csv("/Volumes/workspace/pdge(saber-11)/saber11/BRONZE/trabajo.csv")

In [0]:
print("Rows: ", df.count())
print("Columns: ", len(df.columns))
df.display()

## Reporte de Calidad de Datos

#### Tipos de Datos

In [0]:
df.printSchema()

Renombramiento de columnas

In [0]:
RENAME_COLS = {
    # Identificación
    "DIRECTORIO": "ID_DIRECTORIO",  # NUM
    "SECUENCIA_P": "ID_SECUENCIA_PERSONA",  # NUM
    "TAM_HOGAR": "TAM_HOGAR",  # NUM
    "ORDEN": "ORDEN",  # NUM
    
    # Actividad económica
    "RAMA2D": "RAMA2D",  # NUM
    "RAMA4D": "RAMA4D",  # NUM
    "CLASE": "ZONA",  # ENUM(1. Cabecera, 2. Resto)
    "OFICIO": "CODIGO_OFICIO",  # NUM
    
    # Actividad principal
    "P400": "ACTIVIDAD_PRINCIPAL_SEMANA",  # ENUM(1. Trabajando, 2. Buscando trabajo, 3. Estudiando, 4. OFICIOs del hogar, 5. Incapacitado permanente para trabajar, 6. Otra actividad)
    "P401": "TRABAJO_PAGO_SEMANA",  # BOOL
    "P402": "TRABAJO_SEMANA",  # BOOL
    "P403": "TRABAJO_NO_PAGO",  # BOOL
    "P404": "BUSCA_TRABAJO",  # BOOL
    
    # Actividades no remuneradas (P421)
    "P421S1": "AYUDA_CAMPO",  # BOOL
    "P421S1A1": "HORAS_AYUDA_CAMPO",  # NUM
    "P421S2": "OFICIOS_HOGAR",  # BOOL
    "P421S2A1": "HORAS_OFICIOS_HOGAR",  # NUM
    "P421S3": "OFICIOS_OTROS_HOGARES",  # BOOL
    "P421S3A1": "HORAS_OFICIOS_OTROS_HOGARES",  # NUM
    "P421S4": "CUIDADO_NINOS",  # BOOL
    "P421S4A1": "HORAS_CUIDADO_NINOS",  # NUM
    "P421S5": "CUIDADO_ENFERMOS",  # BOOL
    "P421S5A1": "HORAS_CUIDADO_ENFERMOS",  # NUM
    "P421S6": "ELABORACION_PRENDAS",  # BOOL
    "P421S6A1": "HORAS_ELABORACION_PRENDAS",  # NUM
    "P421S7": "ASISTENCIA_CURSOS",  # BOOL
    "P421S7A1": "HORAS_ASISTENCIA_CURSOS",  # NUM
    "P421S8": "AUTOCONSTRUCCION",  # BOOL
    "P421S8A1": "HORAS_AUTOCONSTRUCCION",  # NUM
    "P421S9": "TRABAJO_COMUNITARIO",  # BOOL
    "P421S9A1": "HORAS_TRABAJO_COMUNITARIO",  # NUM
    "P421S10": "ACTIVIDADES_CIVICAS",  # BOOL
    "P421S10A1": "HORAS_ACTIVIDADES_CIVICAS",  # NUM
    
    # Oficios del hogar (P422)
    "P422S1": "OFICIO_LAVAR",  # BOOL
    "P422S2": "OFICIO_PLANCHAR",  # BOOL
    "P422S3": "OFICIO_COCINAR",  # BOOL
    "P422S4": "OFICIO_CUIDAR_PERSONAS",  # BOOL
    "P422S5": "OFICIO_HUERTA_ANIMALES",  # BOOL
    "P422S6": "OFICIO_MANDADOS",  # BOOL
    "P422S7": "OFICIO_LIMPIEZA",  # BOOL
    "P422S8": "OFICIO_OTROS",  # BOOL
    
    # Motivación
    "P423": "RAZON_OFICIOS",  # ENUM,(1. Sus padres tienen que trabajar, 2. No hay otra persona quien los haga, 3. Tiene que colaborar en el hogar, 4. Debe aprender a hacerlos, 5. Por herencia o tradición, 6. Otra razón )
    "P405": "RAZON_TRABAJO",  # ENUM(1. Debe costearse el estudio, 2. Debe ayudar con los gastos de la casa, 3. Debe participar en la actividad económica de la familia, 4. El trabajo lo forma y lo hace honrado, 5. El trabajo lo aleja de los vicios, 6. Le gusta trabajar para tener su propio dinero, 7. Otra razón)
    "P408": "CARGO_TRABAJO",  # ENUM(1. Obrero o empleado de empresa particular, 2. Obrero o empleado del gobierno, 3. Empleado doméstico, 4. Trabajador por cuenta propia, 5. Patrón o empleador, 6. Trabajador familiar sin remuneración, 7. Trabajador familiar sin remuneración en empresas o negocios de otros hogares, 8. Jornalero o peón, 9. Otro)
    
    # Ingresos / condiciones laborales
    "P409": "INGRESO_MENSUAL", # NUM
    "P410": "RECIBE_ALIMENTOS", # BOOL
    "P410S1": "VALOR_ALIMENTOS", # NUM
    "P411": "RECIBE_VIVIENDA", # BOOL
    "P411S1": "VALOR_VIVIENDA", # NUM
    "P412": "RECIBE_PAGO_ESPECIE", # BOOL
    "P412S1": "VALOR_PAGO_ESPECIE", # NUM
    "P413": "INGRESO_TOTAL", # NUM
    "P414": "MESES_TRABAJADOS", # NUM
    "P415": "HORAS_SEMANALES", # NUM
    "P416": "HORAS_SEMANALES_TRABAJADAS", # NUM
    "P420": "UBICACION_TRABAJO", # ENUM(1. En esta vivienda, 2. En otras viviendas, 3. En kiosko - caseta, 4. En un vehículo, 5. De puerta en puerta, 6. Sitio al descubierto en la calle (ambulante y estacionario), 7. Local fijo, oficina, fábrica, etc., 8. En el campo o área rural, mar o río, 9. En una obra en construcción, 10. En una mina o cantera, 11. Otro)
    
    # Ubicación / expansión
    "AREA": "AREA", # ENUM(05 Medellín A.M 08 Barranquilla A.M 11 Bogotá D.C 13 Cartagena 15 Tunja 17 Manizales A.M 18 Florencia 19 Popayán 20 Valledupar 23 Montería 27 Quibdó 41 Neiva 44 Riohacha 47 Santa Marta 50 Villavicencio 52 Pasto 54 Cúcuta A.M 63 Armenia 66 Pereira A.M 68 Bucaramanga A.M 70 Sincelejo 73 Ibagué 76 Cali A.M )
    "FEX_CT": "FACTOR_EXPANSION", # NUM
}

for old, new in RENAME_COLS.items():
    if old in df.columns:
        df = df.withColumnRenamed(old, new)

Casteo de datos

In [0]:
"""
GRUPOS DE COLUMNAS
"""
BOOL_PURO = [
    "TRABAJO_PAGO_SEMANA", "TRABAJO_SEMANA", "TRABAJO_NO_PAGO",
    "BUSCA_TRABAJO",
    "AYUDA_CAMPO", "OFICIOS_HOGAR", "OFICIOS_OTROS_HOGARES",
    "CUIDADO_NINOS", "CUIDADO_ENFERMOS", "ELABORACION_PRENDAS",
    "ASISTENCIA_CURSOS", "AUTOCONSTRUCCION", "TRABAJO_COMUNITARIO",
    "ACTIVIDADES_CIVICAS",
    "OFICIO_LAVAR", "OFICIO_PLANCHAR", "OFICIO_COCINAR",
    "OFICIO_CUIDAR_PERSONAS", "OFICIO_HUERTA_ANIMALES",
    "OFICIO_MANDADOS", "OFICIO_LIMPIEZA", "OFICIO_OTROS"
]
BOOL_EXTENDIDO = [
    "RECIBE_ALIMENTOS",   # 1,2,3
    "RECIBE_VIVIENDA",    # 1,2,3
    "RECIBE_PAGO_ESPECIE" # 1,2,3
]

INT_COLS = [
    "ID_SECUENCIA_PERSONA", "ORDEN", "MESES_TRABAJADOS",
    "HORAS_SEMANALES", "HORAS_SEMANALES_TRABAJADAS",
    "TAM_HOGAR"
]

# Horas
INT_COLS += [c for c in df.columns if "HORAS_" in c]

LONG_COLS = [
    "ID_DIRECTORIO", "INGRESO_MENSUAL", "VALOR_ALIMENTOS",
    "VALOR_VIVIENDA", "VALOR_PAGO_ESPECIE", "INGRESO_TOTAL",
    "FACTOR_EXPANSION"
]

ENUM_COLS = [
    "ZONA", "ACTIVIDAD_PRINCIPAL_SEMANA", "RAZON_OFICIOS",
    "RAZON_TRABAJO", "CARGO_TRABAJO", "UBICACION_TRABAJO"
]

STRING_COLS = ["RAMA2D", "RAMA4D", "AREA"]



In [0]:
"""""""""
CASTEOS
"""""""""
df00 = df

# BOOL
def cast_bool(col):
    return (
        F.when(F.col(col) == "1", True)
         .when(F.col(col) == "2", False)
    )
for c in BOOL_PURO:
    if c in df00.columns:
        df00 = df00.withColumn(c, cast_bool(c))

# INT
for c in INT_COLS + BOOL_EXTENDIDO:
    if c in df00.columns:
        df00 = df00.withColumn(c, F.col(c).try_cast(IntegerType()))

# LONG
for c in LONG_COLS:
    if c in df00.columns:
        df00 = df00.withColumn(c, F.col(c).try_cast(LongType()))

# ENUM → INT
for c in ENUM_COLS:
    if c in df00.columns:
        df00 = df00.withColumn(c, F.col(c).try_cast(IntegerType()))

# STRING
for c in STRING_COLS:
    if c in df00.columns:
        df00 = df00.withColumn(c, F.col(c).try_cast(StringType()))

# Normalizacion de datos
NULL_LIKE = ["", " ", "NULL", "null", "NA"]
for c in STRING_COLS:
    if c in df00.columns:
        df00 = df00.withColumn(c, F.when(F.col(c).isin(NULL_LIKE), None).otherwise(F.col(c)))

Dataframes de ENUMS

In [0]:
# --------------------------------------------------
# 1. ZONA (antes CLASE)
# 1 = Cabecera, 2 = Resto
# --------------------------------------------------
dim_zona = spark.createDataFrame([
    (1, "Cabecera"),
    (2, "Resto"),
], ["id_zona", "desc_zona"])


# --------------------------------------------------
# 2. ACTIVIDAD_PRINCIPAL_SEMANA (P400)
# --------------------------------------------------
dim_actividad_principal_semana = spark.createDataFrame([
    (1, "Trabajando"),
    (2, "Buscando trabajo"),
    (3, "Estudiando"),
    (4, "Oficios del hogar"),
    (5, "Incapacitado permanente para trabajar"),
    (6, "Otra actividad"),
], ["id_actividad_principal_semana", "desc_actividad_principal_semana"])


# --------------------------------------------------
# 3. RAZON_OFICIOS (P423)
# --------------------------------------------------
dim_razon_oficios = spark.createDataFrame([
    (1, "Sus padres tienen que trabajar"),
    (2, "No hay otra persona quien los haga"),
    (3, "Tiene que colaborar en el hogar"),
    (4, "Debe aprender a hacerlos"),
    (5, "Por herencia o tradición"),
    (6, "Otra razón"),
], ["id_razon_oficios", "desc_razon_oficios"])


# --------------------------------------------------
# 4. RAZON_TRABAJO (P405)
# --------------------------------------------------
dim_razon_trabajo = spark.createDataFrame([
    (1, "Debe costearse el estudio"),
    (2, "Debe ayudar con los gastos de la casa"),
    (3, "Debe participar en la actividad económica de la familia"),
    (4, "El trabajo lo forma y lo hace honrado"),
    (5, "El trabajo lo aleja de los vicios"),
    (6, "Le gusta trabajar para tener su propio dinero"),
    (7, "Otra razón"),
], ["id_razon_trabajo", "desc_razon_trabajo"])


# --------------------------------------------------
# 5. CARGO_TRABAJO (P408)
# --------------------------------------------------
dim_cargo_trabajo = spark.createDataFrame([
    (1, "Obrero o empleado de empresa particular"),
    (2, "Obrero o empleado del gobierno"),
    (3, "Empleado doméstico"),
    (4, "Trabajador por cuenta propia"),
    (5, "Patrón o empleador"),
    (6, "Trabajador familiar sin remuneración"),
    (7, "Trabajador familiar sin remuneración en empresas o negocios de otros hogares"),
    (8, "Jornalero o peón"),
    (9, "Otro"),
], ["id_cargo_trabajo", "desc_cargo_trabajo"])


# --------------------------------------------------
# 6. UBICACION_TRABAJO (P420)
# --------------------------------------------------
dim_ubicacion_trabajo = spark.createDataFrame([
    (1, "En esta vivienda"),
    (2, "En otras viviendas"),
    (3, "En kiosko - caseta"),
    (4, "En un vehículo"),
    (5, "De puerta en puerta"),
    (6, "Sitio al descubierto en la calle (ambulante y estacionario)"),
    (7, "Local fijo, oficina, fábrica, etc."),
    (8, "En el campo o área rural, mar o río"),
    (9, "En una obra en construcción"),
    (10, "En una mina o cantera"),
    (11, "Otro"),
], ["id_ubicacion_trabajo", "desc_ubicacion_trabajo"])


# --------------------------------------------------
# 7. AREA
# Ojo: en el diccionario AREA es character, así que dejo id como string
# --------------------------------------------------
dim_area = spark.createDataFrame([
    ("05", "Medellín A.M"),
    ("08", "Barranquilla A.M"),
    ("11", "Bogotá D.C"),
    ("13", "Cartagena"),
    ("15", "Tunja"),
    ("17", "Manizales A.M"),
    ("18", "Florencia"),
    ("19", "Popayán"),
    ("20", "Valledupar"),
    ("23", "Montería"),
    ("27", "Quibdó"),
    ("41", "Neiva"),
    ("44", "Riohacha"),
    ("47", "Santa Marta"),
    ("50", "Villavicencio"),
    ("52", "Pasto"),
    ("54", "Cúcuta A.M"),
    ("63", "Armenia"),
    ("66", "Pereira A.M"),
    ("68", "Bucaramanga A.M"),
    ("70", "Sincelejo"),
    ("73", "Ibagué"),
    ("76", "Cali A.M"),
], ["id_area", "desc_area"])


# --------------------------------------------------
# 8. BOOL_EXTENDIDO
# Se dejan como INT y se traducen con dimensión
# Aplica para:
# RECIBE_ALIMENTOS (P410)
# RECIBE_VIVIENDA (P411)
# RECIBE_PAGO_ESPECIE (P412)
# --------------------------------------------------
dim_respuesta_si_no_ns_nr = spark.createDataFrame([
    (1, "Sí"),
    (2, "No"),
    (3, "No sabe / No informa"),
], ["id_respuesta", "desc_respuesta"])

In [0]:
df00.printSchema()

#### Valores nulos

In [0]:
null_count = df00.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df00.columns])
print("Columnas con nulos:")
for c in null_count.columns:
    count = null_count.select(c).collect()[0][0]
    if count > 0:
        print(c, f'({count})')


#### Identificación de Duplicados

In [0]:
duplicates = df00.groupBy(df00.columns).count().filter("count > 1")
print("Cantidad de duplicados: ", duplicates.count())

## Limpieza y Transformaciones

#### Eliminacion de columnas irrelevantes

In [0]:
COLS_TO_DROP = ["ID_DIRECTORIO", "ID_SECUENCIA_PERSONA", "ORDEN", "RAMA2D", "RAMA4D", "CODIGO_OFICIO", "FACTOR_EXPANSION"]
df00 = df00.drop(*COLS_TO_DROP)

#### Agregacion de columnas

Oficios del hogar

Reducir las columnas de oficios a:
 - TOTAL_TIPOS_OFICIO_HOGAR
 - REALIZA_ALGUN_OFICIO_HOGAR

In [0]:
df01 = df00
OFICIO_COLS = [
    "OFICIO_LAVAR",
    "OFICIO_PLANCHAR",
    "OFICIO_COCINAR",
    "OFICIO_CUIDAR_PERSONAS",
    "OFICIO_HUERTA_ANIMALES",
    "OFICIO_MANDADOS",
    "OFICIO_LIMPIEZA",
    "OFICIO_OTROS",
]

# Cantidad de tipos de oficio realizados
df01 = df00.withColumn(
    "TOTAL_TIPOS_OFICIO_HOGAR",
    sum(
        F.when(F.col(c) == True, F.lit(1)).otherwise(F.lit(0))
        for c in OFICIO_COLS if c in df00.columns
    )
)

# Indicador consolidado de si realizo al menos un oficio del hogar
df01 = df01.withColumn(
    "REALIZA_ALGUN_OFICIO_HOGAR",
    F.when(F.col("TOTAL_TIPOS_OFICIO_HOGAR") > 0, True).otherwise(False)
)

df01 = df01.withColumnRenamed( "HORAS_OFICIOS_HOGAR","TOTAL_HORAS_OFICIO_HOGAR")

df01 = df01.drop(*OFICIO_COLS)

Actividades no remuneradas

Reducir las columnas de actividades no remuneradas a:
- TOTAL_ACT_NO_REMUNERADAS
- TOTAL_HORAS_NO_REMUNERADAS
- REALIZA_ACTIVIDAD_NO_REMUNERADA

In [0]:
ACT_NO_REM_COLS = [
    ("AYUDA_CAMPO", "HORAS_AYUDA_CAMPO"),
    ("OFICIOS_OTROS_HOGARES", "HORAS_OFICIOS_OTROS_HOGARES"),
    ("CUIDADO_NINOS", "HORAS_CUIDADO_NINOS"),
    ("CUIDADO_ENFERMOS", "HORAS_CUIDADO_ENFERMOS"),
    ("ELABORACION_PRENDAS", "HORAS_ELABORACION_PRENDAS"),
    ("ASISTENCIA_CURSOS", "HORAS_ASISTENCIA_CURSOS"),
    ("AUTOCONSTRUCCION", "HORAS_AUTOCONSTRUCCION"),
    ("TRABAJO_COMUNITARIO", "HORAS_TRABAJO_COMUNITARIO"),
    ("ACTIVIDADES_CIVICAS", "HORAS_ACTIVIDADES_CIVICAS"),
]

present_pairs = [
    (flag, hours)
    for flag, hours in ACT_NO_REM_COLS
    if flag in df01.columns and hours in df01.columns
]

df02 = df01
# TOTAL TIPOS
df02 = df02.withColumn(
    "TOTAL_ACT_NO_REMUNERADAS",
    sum(
        F.when(F.col(flag) == True, 1).otherwise(0)
        for flag, _ in present_pairs
    ) if present_pairs else F.lit(0)
)

# TOTAL HORAS 
df02 = df02.withColumn(
    "TOTAL_HORAS_NO_REMUNERADAS",
    sum(
        F.when(
            F.col(flag) == True,
            F.coalesce(F.col(hours), F.lit(0))  # NULL → 0
        ).otherwise(0)
        for flag, hours in present_pairs
    ) if present_pairs else F.lit(0)
)

# FLAG
df02 = df02.withColumn(
    "REALIZA_ACTIVIDAD_NO_REMUNERADA",
    F.col("TOTAL_ACT_NO_REMUNERADAS") > 0
)

# DROP columnas originales
df02 = df02.drop(*[c for pair in present_pairs for c in pair])

Ingresos

Reducir las columnas de ingresos a:
- INGRESO_MENSUAL
- INGRESOS_NO_MONETARIOS
- TOTAL_INGRESO_MENSUAL (INGRESO_MENSUAL + INGRESOS_NO_MONETARIOS)


In [0]:
df03 = df02

INGRESO_ESPECIE_COLS = [
    ("RECIBE_ALIMENTOS", "VALOR_ALIMENTOS"),
    ("RECIBE_VIVIENDA", "VALOR_VIVIENDA"),
    ("RECIBE_PAGO_ESPECIE", "VALOR_PAGO_ESPECIE"),
]

present_pairs = [(flag, val) for flag, val in INGRESO_ESPECIE_COLS if flag in df03.columns and val in df03.columns]

# Total ingresos no monetarios
df03 = df03.withColumn(
    "INGRESOS_NO_MONETARIOS",
    reduce(
        lambda acc, pair: acc + F.when(
            F.col(pair[0]) == 1,  # solo si respondió "Sí"
            F.coalesce(F.col(pair[1]), F.lit(0))
        ).otherwise(0),
        present_pairs[1:],
        F.when(
            F.col(present_pairs[0][0]) == 1,
            F.coalesce(F.col(present_pairs[0][1]), F.lit(0))
        ).otherwise(0)
    ) if present_pairs else F.lit(0)
)

# Eliminacion de nulos de ingreso mensual
df03 = df03.withColumn(
    "INGRESO_MENSUAL",
    F.when(F.col("INGRESO_MENSUAL").isNull(), 0)
     .otherwise(F.col("INGRESO_MENSUAL"))
)
df03 = df03.withColumn(
    "INGRESO_TOTAL",
    F.when(F.col("INGRESO_TOTAL").isNull(), 0)
     .otherwise(F.col("INGRESO_TOTAL"))
)

df03 = df03.withColumn("TOTAL_INGRESOS_MENSUALES", F.col("INGRESO_MENSUAL") + F.col("INGRESOS_NO_MONETARIOS"))

df03 = df03.drop(*[pair[0] for pair in present_pairs])
df03 = df03.drop(*[pair[1] for pair in present_pairs])
df03 = df03.drop("INGRESO_MENSUAL")
df03 = df03.withColumnRenamed("INGRESO_TOTAL", "INGRESO_TOTAL_REPORTADO")

HORAS trabajadas y carga total de trabajo (HORAS no remuneradas + HORAS trabajadas)

In [0]:
df04 = df03

HOUR_COLS = [
    "TOTAL_HORAS_OFICIO_HOGAR",
    "HORAS_SEMANALES",
    "HORAS_SEMANALES_TRABAJADAS",
    "TOTAL_HORAS_NO_REMUNERADAS",
]

for c in HOUR_COLS:
    if c in df04.columns:
        df04 = df04.withColumn(c, F.coalesce(F.col(c), F.lit(0)))

df04 = df04.withColumn(
    "CARGA_TOTAL_TRABAJO",
    F.col("HORAS_SEMANALES")
    + F.col("TOTAL_HORAS_OFICIO_HOGAR")
    + F.col("TOTAL_HORAS_NO_REMUNERADAS")
)

df04 = df04.drop("HORAS_SEMANALES_TRABAJADAS")


Estado laboral infantil 

In [0]:
df05 = df04

BOOL_COLS = [
    "TRABAJO_PAGO_SEMANA",
    "TRABAJO_SEMANA",
    "TRABAJO_NO_PAGO",
    "BUSCA_TRABAJO",
    "OFICIOS_HOGAR",
    "REALIZA_ALGUN_OFICIO_HOGAR",
    "REALIZA_ACTIVIDAD_NO_REMUNERADA"
]
# Imputacion de nulos
for c in BOOL_COLS:
    if c in df05.columns:
        df05 = df05.withColumn(c, F.coalesce(F.col(c), F.lit(False)))

# Participación laboral general
df05 = df05.withColumn(
    "PARTICIPA_MERCADO_LABORAL",
    F.col("TRABAJO_PAGO_SEMANA") | F.col("TRABAJO_SEMANA") | F.col("TRABAJO_NO_PAGO") | F.col("BUSCA_TRABAJO")
)

# Estado laboral resumido
df05 = df05.withColumn(
    "ESTADO_LABORAL_INFANTIL",
    F.when(F.col("TRABAJO_PAGO_SEMANA"), F.lit(1))      # trabajo pago reciente
     .when(F.col("TRABAJO_SEMANA"), F.lit(2))           # tenía trabajo / negocio
     .when(F.col("TRABAJO_NO_PAGO"), F.lit(3))          # trabajo no pago
     .when(F.col("BUSCA_TRABAJO"), F.lit(4))            # buscando trabajo
     .otherwise(F.lit(0))                               # N/A
)

df05 = df05.drop(*["TRABAJO_PAGO_SEMANA", "TRABAJO_SEMANA", "TRABAJO_NO_PAGO", "BUSCA_TRABAJO"])

dim_estado_laboral_infantil = spark.createDataFrame([
    (0, "N/A"),
    (1, "Trabajo pago semana"),
    (2, "Tenía trabajo o negocio"),
    (3, "Trabajo no pago"),
    (4, "Buscando trabajo"),
], ["id_estado_laboral_infantil", "desc_estado_laboral_infantil"])

Imputaciones a 0 por columnas relacionadas con trabajo

In [0]:
ENUM_COLS = [
    "ZONA",
    "ACTIVIDAD_PRINCIPAL_SEMANA",
    "RAZON_OFICIOS",
    "RAZON_TRABAJO",
    "CARGO_TRABAJO",
    "UBICACION_TRABAJO",
]
for c in ENUM_COLS:
    if c in df05.columns:
        df05 = df05.withColumn(c, F.coalesce(F.col(c), F.lit(0)))

NUM_ZERO_COLS = ["MESES_TRABAJADOS"]
for c in NUM_ZERO_COLS:
    if c in df05.columns:
        df05 = df05.withColumn(c, F.coalesce(F.col(c), F.lit(0)))

##### Eliminacion de registros de AREA null

In [0]:
df06 = df05
df06 = df06.dropna(subset=["AREA"])

In [0]:
df_final = df06

print("Rows: ", df_final.count())
print("Columns: ", len(df_final.columns))
df_final.printSchema()

null_count = df_final.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_final.columns])
print("Columnas con nulos:")
for c in null_count.columns:
    count = null_count.select(c).collect()[0][0]
    if count > 0:
        print(c, f'({count})')

df_final.display()

In [0]:
# Descriptiva
# =========================================
# DESCRIPTIVA GENERAL: TRABAJO INFANTIL Y CARGA DE TRABAJO
# =========================================

from pyspark.sql import functions as F
import matplotlib.pyplot as plt
from pyspark.sql.window import Window

df = df_final

# -----------------------------------------
# 1. VALIDACIÓN BÁSICA
# -----------------------------------------
print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()

# -----------------------------------------
# 2. DESCRIPTIVA GENERAL DE VARIABLES NUMÉRICAS
# -----------------------------------------
num_cols = [
    "TAM_HOGAR",
    "TOTAL_HORAS_OFICIO_HOGAR",
    "INGRESO_TOTAL_REPORTADO",
    "MESES_TRABAJADOS",
    "HORAS_SEMANALES",
    "TOTAL_TIPOS_OFICIO_HOGAR",
    "TOTAL_ACT_NO_REMUNERADAS",
    "TOTAL_HORAS_NO_REMUNERADAS",
    "INGRESOS_NO_MONETARIOS",
    "TOTAL_INGRESOS_MENSUALES",
    "CARGA_TOTAL_TRABAJO"
]

print("\n=== Descriptiva general de variables numéricas ===")
df.select(*[c for c in num_cols if c in df.columns]).describe().show(truncate=False)

# -----------------------------------------
# 3. TABLA DE FRECUENCIAS DE VARIABLES CLAVE
# -----------------------------------------
cat_cols = [
    "ZONA",
    "ACTIVIDAD_PRINCIPAL_SEMANA",
    "AREA",
    "PARTICIPA_MERCADO_LABORAL",
    "REALIZA_ALGUN_OFICIO_HOGAR",
    "REALIZA_ACTIVIDAD_NO_REMUNERADA",
    "ESTADO_LABORAL_INFANTIL"
]

for c in [col for col in cat_cols if col in df.columns]:
    print(f"\n=== Frecuencias: {c} ===")
    (
        df.groupBy(c)
          .count()
          .withColumn("pct", F.col("count") / F.lit(df.count()))
          .orderBy(F.desc("count"))
          .show(truncate=False)
    )

# -----------------------------------------
# 4. TABLA DESCRIPTIVA POR ESTADO LABORAL INFANTIL
# -----------------------------------------
print("\n=== Perfil por ESTADO_LABORAL_INFANTIL ===")
estado_desc = (
    df.groupBy("ESTADO_LABORAL_INFANTIL")
      .agg(
          F.count("*").alias("n"),
          F.avg("CARGA_TOTAL_TRABAJO").alias("prom_carga_total"),
          F.avg("HORAS_SEMANALES").alias("prom_horas_semanales"),
          F.avg("TOTAL_HORAS_OFICIO_HOGAR").alias("prom_horas_oficio_hogar"),
          F.avg("TOTAL_HORAS_NO_REMUNERADAS").alias("prom_horas_no_remuneradas"),
          F.avg("TOTAL_INGRESOS_MENSUALES").alias("prom_ingresos_mensuales"),
          F.avg(F.col("PARTICIPA_MERCADO_LABORAL").cast("int")).alias("pct_participa_mercado"),
          F.avg(F.col("REALIZA_ALGUN_OFICIO_HOGAR").cast("int")).alias("pct_realiza_oficios"),
          F.avg(F.col("REALIZA_ACTIVIDAD_NO_REMUNERADA").cast("int")).alias("pct_act_no_remunerada")
      )
      .orderBy("ESTADO_LABORAL_INFANTIL")
)

estado_desc.show(truncate=False)

# -----------------------------------------
# 5. TABLA POR ZONA
# -----------------------------------------
print("\n=== Perfil por ZONA ===")
zona_desc = (
    df.groupBy("ZONA")
      .agg(
          F.count("*").alias("n"),
          F.avg("CARGA_TOTAL_TRABAJO").alias("prom_carga_total"),
          F.avg("HORAS_SEMANALES").alias("prom_horas_semanales"),
          F.avg("TOTAL_HORAS_OFICIO_HOGAR").alias("prom_horas_oficio_hogar"),
          F.avg("TOTAL_HORAS_NO_REMUNERADAS").alias("prom_horas_no_remuneradas"),
          F.avg("TOTAL_INGRESOS_MENSUALES").alias("prom_ingresos_mensuales"),
          F.avg(F.col("PARTICIPA_MERCADO_LABORAL").cast("int")).alias("pct_participa_mercado")
      )
      .orderBy("ZONA")
)

zona_desc.show(truncate=False)

# -----------------------------------------
# 6. TABLA POR AREA
# -----------------------------------------
print("\n=== Perfil por AREA ===")
area_desc = (
    df.groupBy("AREA")
      .agg(
          F.count("*").alias("n"),
          F.avg("CARGA_TOTAL_TRABAJO").alias("prom_carga_total"),
          F.avg("HORAS_SEMANALES").alias("prom_horas_semanales"),
          F.avg("TOTAL_HORAS_OFICIO_HOGAR").alias("prom_horas_oficio_hogar"),
          F.avg("TOTAL_HORAS_NO_REMUNERADAS").alias("prom_horas_no_remuneradas"),
          F.avg("TOTAL_INGRESOS_MENSUALES").alias("prom_ingresos_mensuales"),
          F.avg(F.col("PARTICIPA_MERCADO_LABORAL").cast("int")).alias("pct_participa_mercado")
      )
      .orderBy("AREA")
)

area_desc.show(truncate=False)

# -----------------------------------------
# 7. TABLA CRUZADA: ZONA x ESTADO LABORAL INFANTIL
# -----------------------------------------
print("\n=== Distribución de ESTADO_LABORAL_INFANTIL por ZONA ===")
zona_estado = (
    df.groupBy("ZONA", "ESTADO_LABORAL_INFANTIL")
      .count()
      .withColumn(
          "pct_dentro_zona",
          F.col("count") / F.sum("count").over(Window.partitionBy("ZONA"))
      )
      .orderBy("ZONA", "ESTADO_LABORAL_INFANTIL")
)

zona_estado.show(truncate=False)

# -----------------------------------------
# 8. GRÁFICA 1: CARGA TOTAL DE TRABAJO PROMEDIO POR ESTADO LABORAL INFANTIL
# -----------------------------------------
pdf_estado = estado_desc.toPandas()

plt.figure(figsize=(9, 5))
plt.bar(
    pdf_estado["ESTADO_LABORAL_INFANTIL"].astype(str),
    pdf_estado["prom_carga_total"]
)
plt.xlabel("Estado laboral infantil")
plt.ylabel("Carga total de trabajo promedio")
plt.title("Carga total de trabajo promedio por estado laboral infantil")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# -----------------------------------------
# 9. GRÁFICA 2: INGRESOS MENSUALES PROMEDIO POR ZONA
# -----------------------------------------
pdf_zona = zona_desc.toPandas()

plt.figure(figsize=(8, 5))
plt.bar(
    pdf_zona["ZONA"].astype(str),
    pdf_zona["prom_ingresos_mensuales"]
)
plt.xlabel("Zona")
plt.ylabel("Ingresos mensuales promedio")
plt.title("Ingresos mensuales promedio por zona")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()